# Studi Kasus 2 — Benchmark Komputasi dan Pemecahan Masalah Skala Besar
**Materi:** Bridging Global Infrastructure to Address Local Challenge in HPC  
Eksperimen ini menggunakan Monte Carlo untuk memperkirakan π, lalu mendiskusikan pengukuran kinerja dan keterbatasan paralelisme.

## Rumusan masalah dan batasan
Pertanyaan: bagaimana jumlah sampel memengaruhi estimasi dan bagaimana waktu eksekusi berubah ketika pekerjaan dibagi? Notebook ini berjalan di mesin lokal, bukan superkomputer. Hasil benchmark hanya bermakna untuk lingkungan pengujian yang sama.

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

def estimate_pi(n_samples, seed=123):
    rng = np.random.default_rng(seed)
    points = rng.random((n_samples, 2))
    inside = np.sum(np.sum(points**2, axis=1) <= 1.0)
    return 4.0 * inside / n_samples

rows = []
for n in [1_000, 10_000, 100_000, 500_000]:
    start = time.perf_counter()
    estimate = estimate_pi(n, seed=123)
    elapsed = time.perf_counter() - start
    rows.append({"sampel": n, "estimasi_pi": estimate,
                 "galat_absolut": abs(estimate - math.pi), "waktu_detik": elapsed})
results = pd.DataFrame(rows)
display(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(results["sampel"], results["galat_absolut"], marker="o")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Galat absolut vs jumlah sampel")
axes[0].set_xlabel("Jumlah sampel")
axes[0].set_ylabel("Galat absolut")
axes[1].plot(results["sampel"], results["waktu_detik"], marker="o")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Waktu eksekusi")
axes[1].set_xlabel("Jumlah sampel")
axes[1].set_ylabel("Detik")
plt.tight_layout()
plt.show()

## Benchmark pembagian pekerjaan
Fungsi berikut membagi sampel ke beberapa potongan. Ini menunjukkan pola kerja yang bisa dibagi, tetapi penggunaan thread bukan jaminan lebih cepat untuk tugas CPU-bound di Python. Overhead, GIL, ukuran potongan, dan kondisi sistem dapat memengaruhi hasil.

In [ ]:
def chunk_hits(args):
    n, seed = args
    rng = np.random.default_rng(seed)
    pts = rng.random((n, 2))
    return int(np.sum(np.sum(pts**2, axis=1) <= 1.0))

def estimate_pi_chunked(total_samples, chunks=4, seed=500):
    sizes = [total_samples // chunks] * chunks
    for i in range(total_samples % chunks):
        sizes[i] += 1
    jobs = [(size, seed+i) for i, size in enumerate(sizes)]
    with ThreadPoolExecutor(max_workers=chunks) as pool:
        hits = sum(pool.map(chunk_hits, jobs))
    return 4 * hits / total_samples

bench = []
N = 300_000
for chunks in [1, 2, 4, 8]:
    start = time.perf_counter()
    estimate = estimate_pi_chunked(N, chunks=chunks)
    elapsed = time.perf_counter() - start
    bench.append({"potongan": chunks, "estimasi_pi": estimate, "waktu_detik": elapsed})
bench_df = pd.DataFrame(bench)
baseline = bench_df.loc[bench_df["potongan"] == 1, "waktu_detik"].iloc[0]
bench_df["speedup_vs_1"] = baseline / bench_df["waktu_detik"]
display(bench_df)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(bench_df["potongan"], bench_df["waktu_detik"], marker="o")
plt.xlabel("Jumlah potongan / worker")
plt.ylabel("Waktu (detik)")
plt.title("Benchmark lokal: jangan mengasumsikan paralelisme selalu lebih cepat")
plt.show()

## Interpretasi
- Monte Carlo menggunakan sampel acak; galat tidak selalu turun secara monoton pada setiap ukuran sampel.
- Benchmark perlu melaporkan ukuran pekerjaan, seed, jumlah worker, perangkat, dan pengulangan.
- Untuk HPC nyata, ukuran masalah, komunikasi, memori, throughput, dan optimasi algoritma sama pentingnya dengan jumlah core.
- Estimasi π di sini hanya demonstrasi pembagian kerja, bukan simulasi ilmiah berskala besar.